# TFM — Detección de Anomalías en ECG con Big Data
## Sistema de Alertas con Amazon SNS (Anexo D)
**Andrea Ríos López — Master Big Data & Advanced Analytics**

Este notebook implementa la capa de alertas del TFM: consulta los registros anómalos
desde Snowflake, envía notificaciones por email vía Amazon SNS y registra cada alerta
en la tabla `ALERT_HISTORY`.

**Flujo:**
1. Consulta `ANOMALY_SCORES` en Snowflake filtrando `is_anomaly = TRUE`
2. Clasifica la severidad (ALTA / MEDIA / BAJA) según el score
3. Publica la alerta en el tópico SNS `tfm_ecg_anomaly_alerts`
4. Registra la alerta en `ALERT_HISTORY` para trazabilidad

---
## D.1 — Instalación de dependencias

In [ ]:
%pip install snowflake-connector-python pandas --quiet

In [ ]:
dbutils.library.restartPython()

## D.2 — Configuración: Snowflake y SNS

In [ ]:
import snowflake.connector
import boto3
import pandas as pd
from datetime import datetime

# ── Conexión Snowflake ───────────────────────────────────
conn = snowflake.connector.connect(
    user      = 'RIOSLOPEZANDREA',
    password  = '',   # completar antes de ejecutar
    account   = 'elifweh-rg90787',
    warehouse = 'TFM_WH',
    database  = 'TFM_ECG',
    schema    = 'ANOMALY_DETECTION',
    role      = 'ACCOUNTADMIN'
)
print('Snowflake OK')

# ── Cliente Amazon SNS ───────────────────────────────────
ACCESS_KEY = ''  # completar antes de ejecutar
SECRET_KEY = ''  # completar antes de ejecutar
TOPIC_ARN  = 'arn:aws:sns:eu-west-1:123507875529:tfm_ecg_anomaly_alerts'

sns = boto3.client('sns',
    aws_access_key_id     = ACCESS_KEY,
    aws_secret_access_key = SECRET_KEY,
    region_name           = 'eu-west-1'
)
print('SNS OK')

## D.3 — Consulta de registros anómalos desde Snowflake

In [ ]:
cursor = conn.cursor()
cursor.execute("""
    SELECT ecg_id, model_name, anomaly_score, is_anomaly, threshold
    FROM ANOMALY_SCORES
    WHERE is_anomaly = TRUE
    ORDER BY anomaly_score DESC
    LIMIT 10
""")
df_alertas = pd.DataFrame(cursor.fetchall(),
    columns=['ecg_id', 'model_name', 'anomaly_score', 'is_anomaly', 'threshold'])

print(f'Registros anomalos a alertar: {len(df_alertas)}')
display(df_alertas)

## D.4 — Envío de alertas SNS y registro en `ALERT_HISTORY`

La severidad se determina según el score de anomalía:
- **ALTA**: score > 0.25
- **MEDIA**: score entre 0.15 y 0.25
- **BAJA**: score ≤ 0.15

In [ ]:
alertas_enviadas = 0

for _, row in df_alertas.iterrows():
    # Clasificación de severidad
    if row['anomaly_score'] > 0.25:
        severidad = 'ALTA'
    elif row['anomaly_score'] > 0.15:
        severidad = 'MEDIA'
    else:
        severidad = 'BAJA'

    # Construcción del mensaje
    mensaje = f"""
ALERTA TFM - Deteccion de Anomalia ECG
=======================================
ECG ID:        {row['ecg_id']}
Modelo:        {row['model_name']}
Score:         {row['anomaly_score']:.4f}
Umbral:        {row['threshold']:.4f}
Severidad:     {severidad}
Fecha:         {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
=======================================
Sistema de deteccion de anomalias - TFM Andrea Rios Lopez
"""

    # Publicar en SNS
    sns.publish(
        TopicArn = TOPIC_ARN,
        Subject  = f'[TFM-ECG] Anomalia {severidad} — ECG {row["ecg_id"]}',
        Message  = mensaje
    )

    # Registrar en ALERT_HISTORY
    cursor.execute("""
        INSERT INTO ALERT_HISTORY (ecg_id, model_name, severity, alert_date, status)
        VALUES (%s, %s, %s, %s, %s)
    """, (
        int(row['ecg_id']),
        row['model_name'],
        severidad,
        datetime.now(),
        'ENVIADA'
    ))

    alertas_enviadas += 1
    print(f'Alerta enviada — ECG {row["ecg_id"]} | {severidad} | score: {row["anomaly_score"]:.4f}')

conn.commit()
conn.close()
print(f'\nTotal alertas enviadas: {alertas_enviadas}')
print('Revisar Gmail y tabla ALERT_HISTORY en Snowflake')

## D.5 — Verificación en Snowflake

In [ ]:
# Verificar que las alertas quedaron registradas
conn_verify = snowflake.connector.connect(
    user      = 'RIOSLOPEZANDREA',
    password  = '',   # completar antes de ejecutar
    account   = 'elifweh-rg90787',
    warehouse = 'TFM_WH',
    database  = 'TFM_ECG',
    schema    = 'ANOMALY_DETECTION',
    role      = 'ACCOUNTADMIN'
)
cur = conn_verify.cursor()
cur.execute('SELECT * FROM ALERT_HISTORY ORDER BY alert_date DESC')
df_hist = pd.DataFrame(cur.fetchall(),
    columns=['alert_id', 'ecg_id', 'model_name', 'severity', 'alert_date', 'status'])

print(f'Total alertas en ALERT_HISTORY: {len(df_hist)}')
display(df_hist)

cur.close()
conn_verify.close()